# 14. 🚀 Продвинутое: математика обучения с подкреплением

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/14_reinforcement_learning.ipynb)

Код из раздела [modules/14_reinforcement_learning.md](https://github.com/justxor/math-for-ai-2026/blob/main/modules/14_reinforcement_learning.md#m14). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### Value iteration

In [ ]:
import numpy as np
def value_iteration(P, R, gamma=0.9, tol=1e-8):
    """P: (S, A, S) вероятности переходов, R: (S, A) награды."""
    V = np.zeros(P.shape[0])
    while True:
        Q = R + gamma * P @ V                 # (S, A): Беллман для всех пар сразу
        V_new = Q.max(axis=1)
        if np.abs(V_new - V).max() < tol:
            return V_new, Q.argmax(axis=1)    # ценности и жадная политика
        V = V_new

### 🏋️ Практика модуля

**14.2.** Реализуйте Thompson sampling для трёх вариантов с конверсиями 4%, 5%, 6% и посмотрите, куда уходит трафик.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
rng = np.random.default_rng(0)
p_true = np.array([0.04, 0.05, 0.06])
a, b = np.ones(3), np.ones(3)                 # априор Beta(1, 1)
pulls = np.zeros(3, int)
for t in range(20_000):
    k = np.argmax(rng.beta(a, b))             # сэмпл из апостериорного для каждого варианта
    r = rng.random() < p_true[k]
    a[k] += r; b[k] += 1 - r; pulls[k] += 1
print(pulls / pulls.sum())                    # большая часть трафика уходит на 6%

## 🔬 Лаборатория: иллюстрации модуля

Это код картинок из курса. Меняйте параметры (learning rate, температуру, число точек, ядро…) и перезапускайте ячейку — так интуиция появляется быстрее всего.

In [ ]:
# Помощники для иллюстраций (те же, что в scripts/make_figures.py)
import matplotlib.pyplot as plt
from scipy import stats
C = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def save(fig, name):          # в ноутбуке просто показываем картинку
    fig.tight_layout(); plt.show()

def arrow(ax, v, color, label=None, origin=(0, 0), **kw):
    ax.annotate("", xy=(origin[0] + v[0], origin[1] + v[1]), xytext=origin,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, **kw))
    if label:
        ax.text(origin[0] + v[0] * 1.08, origin[1] + v[1] * 1.08, label, color=color, fontsize=13, fontweight="bold")

def rosen_like(x, y):
    return 0.5 * x**2 + 4 * y**2

In [ ]:
def value_iteration():
    H, W, gamma = 5, 6, 0.9
    goal, pit, walls = (0, 5), (1, 5), {(1, 1), (2, 1), (3, 3)}
    V = np.zeros((H, W)); acts = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    def step(s, a):
        r, c = s[0] + a[0], s[1] + a[1]
        if not (0 <= r < H and 0 <= c < W) or (r, c) in walls: return s
        return (r, c)
    for _ in range(100):
        Vn = V.copy()
        for r in range(H):
            for c in range(W):
                s = (r, c)
                if s in walls or s in (goal, pit): continue
                Vn[r, c] = max(-0.04 + gamma * (1.0 if step(s, a) == goal else -1.0 if step(s, a) == pit else V[step(s, a)]) for a in acts)
        V = Vn
    fig, ax = plt.subplots(figsize=(7, 5.2))
    V[goal], V[pit] = 1.0, -1.0
    M = np.ma.array(V, mask=np.zeros_like(V, bool))
    for w in walls: M.mask[w] = True
    im = ax.imshow(M, cmap="RdYlGn", vmin=-1, vmax=1)
    arrows = {(-1, 0): "↑", (1, 0): "↓", (0, -1): "←", (0, 1): "→"}
    for r in range(H):
        for c in range(W):
            s = (r, c)
            if s in walls: ax.text(c, r, "■", ha="center", va="center", fontsize=20, color="gray"); continue
            if s == goal: ax.text(c, r, "+1", ha="center", va="center", fontsize=14, fontweight="bold"); continue
            if s == pit: ax.text(c, r, "−1", ha="center", va="center", fontsize=14, fontweight="bold"); continue
            best = max(acts, key=lambda a: (1.0 if step(s, a) == goal else -1.0 if step(s, a) == pit else V[step(s, a)]))
            ax.text(c, r - 0.15, arrows[best], ha="center", va="center", fontsize=16)
            ax.text(c, r + 0.25, f"{V[r, c]:.2f}", ha="center", va="center", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_title("Value iteration: V(s) и жадная политика (γ = 0.9, штраф за шаг −0.04)")
    fig.colorbar(im, ax=ax, fraction=0.04)
    save(fig, "value_iteration")

value_iteration()